#### This is an explanation of the operations the notebook does
---
The cell bellow makes the necessary imports for the libraries that are going to be used, and establishes a connection with the data lake storage system -minio- by calling the appropriate function that already exists in the configuration file of the framework

In [1]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


### Step 0: Data Ingestion from MinIO

In the cell below, the MinIO object storage is accessed through the client created in the previous step. A specific raw JSON object representing a Google Cloud service is downloaded and stored in the corresponding variable. 

Afterwards, using pandas, this raw JSON is transformed into an initial DataFrame. At this starting point, the DataFrame contains only two top-level key-value pairs:
1. **`skus`**: A nested list containing all the individual services/products (`[list of services]`).
2. **`nextPageToken`**: A string token used for API pagination (`"String"`).

In [2]:
"""
Kubernetes_Engine: 1 page, 1.4 Mib, 1731rows, produces 26 columns
Compute_Engine: 7 pages, 4.1Mib and 1.1Mib, 5000 rows/page, produces 29 columns
Cloud_Storage: 1 page, 1000Kib, 1220 services, 3000 with duplicates, produces 29 columns
Cloud_SQL : 4 pages, produces 29 columns
Networking: 1 page, 1600 services, 2800 non unique, produces 29 columns

"""
object_name = "Compute_Engine/page_1.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


Success. Page loaded and converted into dataframe
Array size: Rows =  5000  and Columns =  2


### Step 1: Loading & Initial Flattening

In the cell below, the first key-value pair is flattened and the list of services is opened, with each service occupying a row. Also, any first-level inner dictionaries (dicts) that a service may contain are also opened.

For example, a nested structure like this:
`category: {key1: value1, key2: value2}`

Opens up and flattens into distinct columns:
`category.key1`, `category.key2` with their respective values mapped across the rows.

In [3]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
df_flat.head(3)
# df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[southamerica-west1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,[southamerica-west1]
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,[europe-west9],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,[europe-west9]
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,[us-west8],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,[us-west8]


### Step 2: Exploding Geographic Regions

In the cell below, we perform an `.explode()` operation on the `geoTaxonomy.regions` column, which originally contains a nested list of locations. 

By exploding this list, each region occupies its own row. This means that for services available in multiple locations, duplicated rows are created for all other attributes, with the only difference being the specific region value in each row.

In [4]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions').reset_index(drop=True)

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[southamerica-west1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,[europe-west9],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,[us-west8],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8


### Step 3: Removing Service Regions

Since the location of the service is provided from the geotaxonomy type and regions the serviceRegions is redundant, and so the column is removed

In [5]:
df_flat3 = df_flat2

if 'serviceRegions' in df_flat3.columns:
    df_flat3.drop(columns=['serviceRegions'], inplace=True)
df_flat3.head()

,name,skuId,description,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2


### Step 4: Exploding Pricing information

Just like the 2 cells above an `.explode()` operation is made to unpack the list of `pricingInfo`, and isolate each of the elements in a single row. For the record, the value within the `pricingInfo` list is a dictionary, which contains the detailed pricing structures and rates for each service

In [6]:
#The only column not fully opened yet is the pricingInfo: structure -> [{key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}}]
df_flat3[['skuId', 'pricingInfo']].head()

#This first explode removes the list: we have now -> {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} 
df_flat4 = df_flat3.explode('pricingInfo').reset_index(drop=True)
df_flat4[['skuId', 'pricingInfo']].head()
df_flat4.head()

,name,skuId,description,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2


### Step 5: Flattening the pricingInfo Dictionary

At this stage of the pipeline, the pricingInfo column contains a dict with the following structure:
`{key1: val1, key2: val2, key3: val3, pricingExpression: {key: val, tieredRates: [{}]}}`

To extract these nested properties into standalone columns, we use the `pd.json_normalize()` again.

**What this operation does:**
1. **Unpacks the Dictionary:** It takes the keys of the `pricingInfo` dictionary (such as `effectiveTime`, `summary`, `currencyConversionRate` and `pricingExpression`) and turns them into separate, clean columns.
2. **Handles Sub-Nested Structures:** If a key contains further nested dictionaries (like `pricingExpression`), it automatically flattens them using dot notation (e.g., `pricingExpression.pricingUnits`, `pricingExpression.baseUnit`).
3. **Preserves Sub-Lists:** Deeply nested lists (like `pricingExpression.tieredRates` which contains the actual price tiers) are kept intact inside their cells, preparing them for the final cleaning steps.

In [7]:
#We are here now: {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} -> we can open the dict with the normalize
#A new dataframe will be created with the pricing info and then concatenated with the original dataframe

pricing1 = pd.json_normalize(df_flat4['pricingInfo'])
pricing1.head()

,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,,1,2026-06-12T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,,1,2026-06-12T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
2,,1,2026-06-12T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,,1,2026-06-12T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
4,,1,2026-06-12T07:00:00Z,GBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN


### Step 6: Aligning Indexes and Merging Data

Since `normalize()` creates a new DataFrame (`pricing1` in our case) we have  to merge it back with our main DataFrame to reconstruct the complete dataset. During this concatenation it is imperative to align the indexes of each line, and drop the pricingInfo column from the first dataset, since it's data will now be in standalone cols


In [8]:
#I now have to concatenate the 2 dataframes beeing carefull though with the indexes
pricing1.index = df_flat4.index

df_flat5 = pd.concat([df_flat4.drop(columns=['pricingInfo']), pricing1], axis=1)
df_flat5.head()
# df_flat5['pricingExpression.tieredRates']

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,...,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,...,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,...,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,...,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,...,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,...,GBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN


### Step 7: Exploding Pricing Tiers

In the cell below, we perform the final `.explode()` operation on the `pricingExpression.tieredRates` column. 
Since a single SKU can have multiple pricing tiers, this operation unpacks the nested list of tiers into individual rows, ensuring every distinct price rate is isolated for analysis.
The value of the list was dictionary/ries so the next operation that is requires is, flattening that dictionary and then concatenation. Those operations take place in the 3 cells bellow -the detailed procedure is not explained as is simillar with above-

In [9]:
df_flat6 = df_flat5.explode('pricingExpression.tieredRates').reset_index(drop=True)

df_flat6[['skuId', 'pricingExpression.tieredRates']].head()

,skuId,pricingExpression.tieredRates
0,0001-B904-8A40,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
1,0001-FC8F-A9AF,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
2,0006-C9C8-BB6F,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
3,0007-4724-5A32,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
4,0007-9388-EF75,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."


In [10]:
pricing2 = pd.json_normalize(df_flat6['pricingExpression.tieredRates'])
pricing2.head()

,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,0.0,USD,0,4954950.0
1,0.0,USD,0,12530000.0
2,0.0,USD,0,20550000.0
3,0.0,USD,0,3815312.0
4,0.0,USD,0,6243470.0


In [11]:
pricing2.index = df_flat6.index

df_final_flat = pd.concat([df_flat6.drop(columns=['pricingExpression.tieredRates']), pricing2], axis=1)
pd.set_option('display.max_columns', None)
df_final_flat.head()

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4954950.0
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,12530000.0
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,20550000.0
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,3815312.0
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,,1,2026-06-12T07:00:00Z,GBy.h,1,gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN,0.0,USD,0,6243470.0


### Step 8: Calculating the Final Price (Handling Units & Nanos)

According to Google Cloud's official documentation, SKU prices are not represented as standard floats. Instead, they are split into two separate components to prevent floating-point precision errors:
* `units`: The whole number part of the price.
* `nanos`: The fractional part of the price, represented as billionths of a cent/dollar ($10^{-9}$).

For example, a price of **$1.75** is stored as `units = 1` and `nanos = 750,000,000`.

**In the cell below, we:**: reconstruct the price by converting both columns to floats, divide `unitPrice.nanos` by $1,000,000,000$, and add them together to create the clean `finalPrice` column.


In [12]:
#The cost of the SKU is units + nanos. For example, a cost of $1.75 is represented as units=1 and nanos=750,000,000. (From google documentation)
#Actions need to be made to create a new column that will contain the final price
df_final_flat['finalPrice'] = df_final_flat['unitPrice.units'].astype(float) + (df_final_flat['unitPrice.nanos'].astype(float) / 1000000000)

df_final_flat[['skuId', 'unitPrice.units', 'unitPrice.nanos', 'finalPrice']].head()

,skuId,unitPrice.units,unitPrice.nanos,finalPrice
0,0001-B904-8A40,0,4954950.0,0.004955
1,0001-FC8F-A9AF,0,12530000.0,0.012530
2,0006-C9C8-BB6F,0,20550000.0,0.020550
3,0007-4724-5A32,0,3815312.0,0.003815
4,0007-9388-EF75,0,6243470.0,0.006243


In [13]:
#Just for debug to search if any line has "1" as unit price
filtered_df = df_final_flat[df_final_flat['unitPrice.units'].astype(float) == 1.0]

filtered_df[['skuId','unitPrice.units', 'unitPrice.nanos', 'finalPrice']].head()

# df_final_flat.head()

,skuId,unitPrice.units,unitPrice.nanos,finalPrice
31,003E-D940-4BC0,1,600000000.0,1.600000
90,008E-3414-1336,1,26867800.0,1.026868
171,0115-21ED-7F03,1,911000000.0,1.911000
444,02D0-3C0A-B826,1,473320560.0,1.473321
445,02D3-A0D2-FC93,1,8000000.0,1.008000


In [14]:
print(df_final_flat['unitPrice.currencyCode'].value_counts())
print ()
print(df_final_flat.shape[0])
print()
print(df_final_flat['skuId'].nunique())

unitPrice.currencyCode
USD    5691
Name: count, dtype: int64

5708

5000


In [15]:
#Just fot debug: filtering lines where the currencny is not USD
eur_rows_df = df_final_flat[df_final_flat['unitPrice.currencyCode'] != 'USD']

print(df_final_flat['unitPrice.currencyCode'].value_counts(dropna=False))

eur_rows_df.head()

#Possible values in category.usageType
df_final_flat['category.usageType'].unique()
# df_final_flat['pricingExpression.usageUnit'].unique()
# df_final_flat['pricingExpression.baseUnit'].unique()

# df_final_flat['pricingExpression.baseUnitDescription'].unique()

unitPrice.currencyCode
USD    5691
NaN      17
Name: count, dtype: int64


array(['OnDemand', 'Preemptible', 'Commit1Yr', 'CmtCudPremium',
       'Commit3Yr'], dtype=object)

### Step 9: Cleaning, Currency normalization, Duplicate removal

**Operations Performed:**
1. We remove any rows where `unitPrice.currencyCode` is absent, filtering out unpriced or incomplete SKU records.

2. We convert all prices into (USD) by dividing the `finalPrice` by the provided `currencyConversionRate`. Easier for future analytics

3. For data profiling purposes we create a separate subset that keeps only the first occurrence of each unique `skuId`. This ensures accurate statistical metrics.


In [16]:
#Throw duplicates, throw records with Nan in currencycode, new column with usd final price

df_clean_no_nan = df_final_flat.dropna(subset=['unitPrice.currencyCode']).reset_index(drop=True).copy() #If the value is nan in this column drop (If the product has no currency registered its problematic)

#Currency normalization
rate = df_clean_no_nan['currencyConversionRate'].astype(float)

#This is our clean array so far
df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate

#Only for profiling tool drop all duplicates
df_for_profiling = df_clean_no_nan.drop_duplicates(subset=['skuId'])

print(f"The size of the array is {df_clean_no_nan.shape[0]} rows, and {df_clean_no_nan.shape[1]}, columns")
print (f"The size of the array used for profiling is {df_for_profiling.shape[0]} rows and {df_for_profiling.shape[1]} columns")
df_clean_no_nan.head(5)

# print(df_for_profiling.shape[0])

The size of the array is 5691 rows, and 28, columns
The size of the array used for profiling is 4983 rows and 28 columns


,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4954950.0,0.004955,0.004955
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,12530000.0,0.012530,0.012530
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,20550000.0,0.020550,0.020550
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,,1,2026-06-12T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,3815312.0,0.003815,0.003815
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,,1,2026-06-12T07:00:00Z,GBy.h,1,gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN,0.0,USD,0,6243470.0,0.006243,0.006243


Πετάμε περιττές στήλες από το dataframe
- Η στήλη pricingExpression.usageUnit λέει την μονάδα μέτρησης της τιμής της υπηρεσίας (πχ 10 δολάριο το μήνα ή την ώρα ή ότι είναι). Δεν χρειάζομαι την στήλη η οποία περιγράφει με λόγια την μονάδα μέτρησης (pricingExpression.usageUnitDescription), px το GBy.h λέει ολογράφως Gigabyte per hour. -> Την πετάω
- Κατά τον ίδιο τρόπο πετάω την στήλη pricingExpression.baseUnitDescription καθώς περιγράφει με λόγια το baseUnit
- Επίσης: στήλε unit, nanos, currencyConversionRate αποδείχθηκαν χρήσιμες για εξαγωγή συμπερασμάτων στα αρχικά στάδια, αλλά πλεόν είναι άχρηστες αφού έχω υπολογίσει την τελική τιμή στο νόμισμα αναφοράς, αλλά και την τελική τιμή σε δολάριο βάση του currencyConversionRate.  -> τις αφαιρώ
- Θα κινηθούμε στο ίδιο πλαίσιο με την azure και αργότερα με την amazon και σε πρώτη φάση θα κρατήσουμε μόνο On-demand η αλλιώς pay as you go υπηρεσίες. Τις άλλες δεν τις θέλω... Άρα μέσω κώδικα κρατάω μόνο όπου category.usageType == 'OnDemand'
- Όμοια για την στήλη summary. Πέρα από τον έλεγχο που γίνεται, στον οποίο διαπιστώθηκε ότι δεν υπάρχει καμία τιμή (είναι κενή), και να υπήρχε δεν μας ενδιαφέρει ιδιαίτερα η περιγραφή της υπηρεσίας αφενός, και αφετέρου στους άλλους παρόχους δεν υπάρχει αντίστοιχη στήλη -> Την πετάω από το τελικό καθαρό Dataframe

In [17]:
# pd.set_option('display.max_rows', None)
df_clean_no_nan.drop(columns=['pricingExpression.usageUnitDescription', 'pricingExpression.baseUnitDescription'], errors='ignore', inplace=True)
df_clean_no_nan.drop(columns=['currencyConversionRate', 'unitPrice.units', 'unitPrice.nanos'], errors='ignore', inplace=True)
df_clean_no_nan = df_clean_no_nan[df_clean_no_nan['category.usageType'] == 'OnDemand'].reset_index(drop=True).copy()
    
# df_clean_no_nan['summary'].unique()
df_clean_no_nan.drop(columns=['summary'], errors='ignore', inplace=True)
df_clean_no_nan.head()
# print (df_clean_no_nan.shape[1])

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,2026-06-12T07:00:00Z,GBy.h,1,By.s,3.600000e+12,NaN,NaN,NaN,0.0,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,NaN,NaN,NaN,0.0,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,NaN,NaN,NaN,0.0,USD,0.005550,0.005550


In [18]:
# from ydata_profiling import ProfileReport

# profile = ProfileReport(df_for_profiling, title="Google Cloud Compute Engine - Unique SKUs Report", explorative=True)

# #Stores in file under the same directory
# profile.to_file("google_billing_unique_analysis.html")

# print("Report created")

**Debug purposes**

In [19]:
df_clean_no_nan['serviceProviderName'].unique()
df_clean_no_nan[df_clean_no_nan['serviceProviderName'] != 'Google'].head()

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,finalPrice,final_price_usd
42,services/6F81-5844-456A/skus/0071-242F-010B,0071-242F-010B,Licensing Fee for Ubuntu Pro FIPS 20.04 LTS (F...,Canonical,Compute Engine,License,Canonical,OnDemand,GLOBAL,NaN,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.035000,0.035000
235,services/6F81-5844-456A/skus/0201-C8D6-9970,0201-C8D6-9970,Licensing Fee for Ubuntu Pro 26.04 LTS (Resolu...,Canonical,Compute Engine,License,Canonical,OnDemand,GLOBAL,NaN,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.122063,0.122063
283,services/6F81-5844-456A/skus/0275-C0AC-E860,0275-C0AC-E860,Licensing Fee for Ubuntu Pro FIPS 22.04 LTS (J...,Canonical,Compute Engine,License,Canonical,OnDemand,GLOBAL,NaN,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,NaN,NaN,NaN,0.0,USD,0.000127,0.000127
360,services/6F81-5844-456A/skus/033E-56F9-C843,033E-56F9-C843,Licensing Fee for Ubuntu Pro 16.04 LTS (Xenial...,Canonical,Compute Engine,License,Canonical,OnDemand,GLOBAL,NaN,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.060548,0.060548
432,services/6F81-5844-456A/skus/03D5-9AC0-9665,03D5-9AC0-9665,Licensing Fee for Ubuntu Minimal 25.04 (Plucky...,Canonical,Compute Engine,License,Canonical,OnDemand,GLOBAL,NaN,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,0.0,USD,0.000000,0.000000


In [20]:
#To check witch of the aggreagation columns actually have a value. Most of them dont
df_clean_no_nan[df_clean_no_nan['aggregationInfo.aggregationCount'].notna()].head()
# df_clean_no_nan['aggregationInfo.aggregationInterval'].unique()

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,finalPrice,final_price_usd
16,services/6F81-5844-456A/skus/003A-21C5-CA77,003A-21C5-CA77,Network Standard Data Transfer Out to Internet...,Google,Compute Engine,Network,StandardInternetEgress,OnDemand,REGIONAL,asia-northeast3,2026-06-12T07:00:00Z,GiBy,1,By,1.073742e+09,ACCOUNT,MONTHLY,1.0,0.0,USD,0.000,0.000
17,services/6F81-5844-456A/skus/003A-21C5-CA77,003A-21C5-CA77,Network Standard Data Transfer Out to Internet...,Google,Compute Engine,Network,StandardInternetEgress,OnDemand,REGIONAL,asia-northeast3,2026-06-12T07:00:00Z,GiBy,1,By,1.073742e+09,ACCOUNT,MONTHLY,1.0,200.0,USD,0.119,0.119
18,services/6F81-5844-456A/skus/003A-21C5-CA77,003A-21C5-CA77,Network Standard Data Transfer Out to Internet...,Google,Compute Engine,Network,StandardInternetEgress,OnDemand,REGIONAL,asia-northeast3,2026-06-12T07:00:00Z,GiBy,1,By,1.073742e+09,ACCOUNT,MONTHLY,1.0,10240.0,USD,0.109,0.109
19,services/6F81-5844-456A/skus/003A-21C5-CA77,003A-21C5-CA77,Network Standard Data Transfer Out to Internet...,Google,Compute Engine,Network,StandardInternetEgress,OnDemand,REGIONAL,asia-northeast3,2026-06-12T07:00:00Z,GiBy,1,By,1.073742e+09,ACCOUNT,MONTHLY,1.0,153600.0,USD,0.097,0.097
51,services/6F81-5844-456A/skus/0086-8707-C3B1,0086-8707-C3B1,Network Inter Region Data Transfer Out from Am...,Google,Compute Engine,Network,InterregionEgress,OnDemand,MULTI_REGIONAL,us-central1,2026-06-12T07:00:00Z,GiBy,1,By,1.073742e+09,ACCOUNT,MONTHLY,1.0,0.0,USD,0.000,0.000


In [21]:
df_clean_no_nan.head()
df_clean_no_nan['startUsageAmount'].unique()

array([0.000e+00, 2.000e+02, 1.024e+04, 1.536e+05, 1.000e+00, 1.024e+03,
       5.000e+00, 3.000e+01])

- Μετά από σκέψη και παραδοχές που φαίνονται στην αναφορά, αποφασίστηκε από την στήλη startUsageAmount μέσω της οποίας δημιουργείται ή έννοια των κλιμακωτών χρεώσεων σχετικά με την παρεχόμενη υπηρεσία, να φιλτράρουμε εν αρχή κρατώντας μόνο τις μηδενικές τιμές, συμβολίζοντας έτσι την διάθεση μας κρατήσουμε μόνο την τιμή βάσης για μία υπηρεσία (πετώντας διπλότυπες εγγραφές βάση χρεώσης) και έπειτα ξέροντας ότι το dataset περιέχει μόνο τιμές βάσης, να πετάξουμε και την ίδια την στήλη στα σκουπίδια
- Επιπλέον μαζί με αυτό, θα πετάξουμε και την τριάδα με τις στήλες aggregation, καθώς αυτές είναι άμεσο επόμενο της στήλης startUsageAmount. Ωστόσο καθώς ορισμένες υπηρεσίες δεν την έιχα εξαρχής, θα είμαστε προσεκτικοί εφαρμόζοντας έναν έλεγχο ύπαρξης εκ των πρωτέρων.

In [22]:
print (f"The tolal rows of the dataset are {df_clean_no_nan.shape[0]}, and columns are {df_clean_no_nan.shape[1]}")

#Αυτό είναι το φίλτρο το οποίο θα μεταφέρω στον parser. Bάζω και έναν έλεγχο για να μην πετάει error
if 'startUsageAmount' in df_clean_no_nan.columns:
    df_clean_no_nan = df_clean_no_nan[df_clean_no_nan['startUsageAmount'] == 0].reset_index(drop=True)

    # H εντολή αυτή ελέγχει αν στην στήλη υπάρχει τιμή διαφορετική 0.0. Βγάζει false, αρα υπάρχει μόνο το 0.0
    (df_clean_no_nan['startUsageAmount'] != 0.0).any()

#Αφού το φίλτρο πέτυχε ήρθε ή ώρα να πετάξω και την στήλη
df_clean_no_nan.drop(columns=['startUsageAmount'], errors='ignore', inplace=True)
print (f"The total rows of the dataset after prunning are {df_clean_no_nan.shape[0]}, and columns are {df_clean_no_nan.shape[1]}")
df_clean_no_nan.head()




The tolal rows of the dataset are 4160, and columns are 22
The total rows of the dataset after prunning are 3704, and columns are 21


,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,NaN,NaN,NaN,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,2026-06-12T07:00:00Z,GBy.h,1,By.s,3.600000e+12,NaN,NaN,NaN,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,NaN,NaN,NaN,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,NaN,NaN,NaN,USD,0.005550,0.005550


In [23]:
df_clean_no_nan['aggregationInfo.aggregationLevel'].unique()

array([nan, 'ACCOUNT'], dtype=object)

In [24]:
# Πετάω τα aggregation αν υπάρχουν 
if 'aggregationInfo.aggregationLevel' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['aggregationInfo.aggregationLevel'], inplace=True)

# 3. Διαγραφή για το aggregationInterval
if 'aggregationInfo.aggregationInterval' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['aggregationInfo.aggregationInterval'], inplace=True)

# 4. Διαγραφή για το aggregationCount
if 'aggregationInfo.aggregationCount' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['aggregationInfo.aggregationCount'], inplace=True)

print (f"The dataset has {df_clean_no_nan.shape[0]} rows, and {df_clean_no_nan.shape[1]} columns")
df_clean_no_nan.head()

The dataset has 3704 rows, and 18 columns


,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,2026-06-12T07:00:00Z,h,1,s,3.600000e+03,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,2026-06-12T07:00:00Z,GBy.h,1,By.s,3.600000e+12,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,2026-06-12T07:00:00Z,GiBy.h,1,By.s,3.865471e+12,USD,0.005550,0.005550


Το effectiveTime μετά από ανάγνωση του documentation και την παραδοχή ότι για την ώρα δεν φτιάχνω κάτι στο οποίο να με ενδιαφέρει η ιστορικότητα, το πετάμε από το τελικό σχήμα το οποίο φτιάχνουμε. (Αντίστοιχα θα το πετάξουμε και από τους άλλου παρόχους)

In [25]:
if 'effectiveTime' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['effectiveTime'], inplace=True)

df_clean_no_nan.head()
# print (f"The dataset has {df_clean_no_nan.shape[1]} columns")

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.baseUnit,pricingExpression.baseUnitConversionFactor,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,h,1,s,3.600000e+03,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,h,1,s,3.600000e+03,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,GBy.h,1,By.s,3.600000e+12,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,GiBy.h,1,By.s,3.865471e+12,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,GiBy.h,1,By.s,3.865471e+12,USD,0.005550,0.005550


##### Commands used for debug purposes

In [26]:
df_clean_no_nan['serviceProviderName'].unique()

array(['Google', 'Canonical', 'FreeBSD', 'Microsoft', 'Fedora CoreOS',
       'SUSE', 'Compute Engine Migration',
       'Dace IT LLC d/b/a Sense Traffic Pulse', 'AlmaLinux'], dtype=object)

In [27]:
df_clean_no_nan['pricingExpression.displayQuantity'].unique()

array([1])

In [28]:
df_clean_no_nan['pricingExpression.baseUnitConversionFactor'].unique()

array([3.60000000e+03, 3.60000000e+12, 3.86547057e+12, 1.07374182e+09,
       2.59200000e+06, 2.78313881e+15])

In [29]:
df_clean_no_nan['pricingExpression.baseUnit'].unique()

array(['s', 'By.s', 'By'], dtype=object)

In [30]:
df_clean_no_nan['pricingExpression.usageUnit'].unique()

array(['h', 'GBy.h', 'GiBy.h', 'GiBy', 'mo', 'GiBy.mo'], dtype=object)

Έχουμε τις εξής στήλες που μας απασχολούν, όλες κάτω από την ομπρέλα του pricingExpression:
- usageUnit -> το κρατάω
- baseUnit
- baseUnitConversionFactor
- displayQuantity
Το ερώτημα είναι ποιες από όλες μου δίνουν όντως χρήσιμη πληροφορία σχετικά με την τιμολόγηση, αν αναλογιστούμε και τις στήλες τις οποίες μέχρι τώρα έχουμε πετάξει. Από αυτές μετά από μελέτη θα κρατήσουμε μόνο την 1η καθώς περίεχει τις μονάδες μέτρησης στις οποίες αντιστοιχίζεται το κόστος, ενώ οι άλλες είναι για εσωετική χρήση από την google και δεν χρησιμεύουν στον χρήστη ο οποίος θα διαβάζει τα δεδομένα, ούτε σε τυχόν στατιστική ανάλυση

In [31]:
print (f"The dataset currently has {df_clean_no_nan.shape[1]} columns.")

The dataset currently has 17 columns.


In [32]:
df_clean_no_nan.head()
if 'pricingExpression.baseUnit' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['pricingExpression.baseUnit'], inplace=True)

if 'pricingExpression.baseUnitConversionFactor' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['pricingExpression.baseUnitConversionFactor'], inplace=True)

if 'pricingExpression.displayQuantity' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['pricingExpression.displayQuantity'], inplace=True)

In [33]:
df_clean_no_nan.head()

,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,pricingExpression.usageUnit,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,h,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,h,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,GBy.h,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,GiBy.h,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,GiBy.h,USD,0.005550,0.005550


In [34]:
print (f"The dataframe has {df_clean_no_nan.shape[1]} cols")
df_clean_no_nan['serviceProviderName'].unique()


The dataframe has 14 cols


array(['Google', 'Canonical', 'FreeBSD', 'Microsoft', 'Fedora CoreOS',
       'SUSE', 'Compute Engine Migration',
       'Dace IT LLC d/b/a Sense Traffic Pulse', 'AlmaLinux'], dtype=object)

In [36]:
print (f"Dimensions (rows, cols): {df_clean_no_nan.shape}")
df_clean_no_nan.head(5)
# pd.set_option('display.max_colwidth', None)
# df_clean_no_nan['description'].head(5)


Dimensions (rows, cols): (3704, 14)


,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,pricingExpression.usageUnit,unitPrice.currencyCode,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,h,USD,0.004955,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,h,USD,0.003815,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,GBy.h,USD,0.006243,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,europe-west12,GiBy.h,USD,0.005080,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-west6,GiBy.h,USD,0.005550,0.005550


##### Στήλη name
- Που χρησιμέυει σε εμένα ? Είναι αυτή το βασικό χαρακτηριστικό της υπηρεσίας ή το skuId? Mήπως το name είναι κάτι αντίστοιχο ,με το rateCode ?
##### Στήλη category.serviceDisplayName
- Το ότι είναι compute engine η υπηρεσία με ενδιαφέρει ; Μάλλον όχι? Αφού το τελικό καθαρό που έχω μόνο compute περιέχει? Ή αυτό ή το αντίστοιχο με το family θα τα απορρίψω. Θα κρατήσω ή το γενικό ή το ειδικό
#### Στήλη usageType
- Το ότι είναι OnDemand το ξέρω...Αυτό φτιάχνω!! Εργαλείο με On - Demand υπηρεσίες!! Άρα ποιος ο λόγος να μέινει; 
***-> Οκ έγινε*** 
#### Στήλη currencyCode
- Όλα είανι σε usd. Άρα ποιος ο λόγος να το κρατάμε. Έχουμε φροντίσει να κάνουμε και την μετατροπή στην τελευταία στήλη, οπότε και η στήλη με το νόμισμα, αλλά και εκείνη που λέει την τιμή απλά, θα την διώξουμε, και θα μετονομάσουμε εκείνη με την τιμή σε priceUSD!! 
***-> Όκ έγινε*** 

In [ ]:
#Την στήλη που λέει ότι είναι OnDemand την απορρίπτουμε
if 'category.usageType' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['category.usageType'], inplace=True)

In [41]:
#Μετονομάζουμε την τελευταία στήλη σε priceUSD όπως την έχει και η aws
df_clean_no_nan.rename(columns={'final_price_usd': 'priceUSD'}, inplace=True)

In [ ]:
#Πετάμε την στήλη με τον τύπο νομίσματος!! Ακόμη και αν ερχόντουσαν τιμές που δεν ήταν δολάριο, έχουμε κάνει την μετατροπή και πλέον όλες είναι
if 'unitPrice.currencyCode' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['unitPrice.currencyCode'], inplace=True)

In [43]:
#Επίσης πετάμε και την στήλη με τον αρχικό υπολογισμό της τιμής. Η τιμή εκεί είναι εκφρασμένη στο νόμισμα που δηλώνει η στήλη με τον τύπο νομίσματος. Άρα καμία χρησιμότητα να μείνει
if 'finalPrice' in df_clean_no_nan.columns:
    df_clean_no_nan.drop(columns=['finalPrice'], inplace=True)

In [47]:
print (f"Dimensions (rows, cols): {df_clean_no_nan.shape}")
df_clean_no_nan.head()

Dimensions (rows, cols): (3704, 11)


,name,skuId,description,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,geoTaxonomy.type,geoTaxonomy.regions,pricingExpression.usageUnit,priceUSD
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,REGIONAL,southamerica-west1,h,0.004955
1,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,Google,Compute Engine,Compute,CPU,REGIONAL,europe-north1,h,0.003815
2,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,Google,Compute Engine,Compute,RAM,REGIONAL,northamerica-northeast2,GBy.h,0.006243
3,services/6F81-5844-456A/skus/000F-0B14-D302,000F-0B14-D302,C3 Sole Tenancy Instance Ram running in Turin,Google,Compute Engine,Compute,RAM,REGIONAL,europe-west12,GiBy.h,0.005080
4,services/6F81-5844-456A/skus/000F-E31B-1D6F,000F-E31B-1D6F,N1 Predefined Instance Ram running in Zurich,Google,Compute Engine,Compute,N1Standard,REGIONAL,europe-west6,GiBy.h,0.005550


In [45]:
df_clean_no_nan['name']

0       services/6F81-5844-456A/skus/0001-B904-8A40
1       services/6F81-5844-456A/skus/0007-4724-5A32
2       services/6F81-5844-456A/skus/0007-9388-EF75
3       services/6F81-5844-456A/skus/000F-0B14-D302
4       services/6F81-5844-456A/skus/000F-E31B-1D6F
                           ...                     
3699    services/6F81-5844-456A/skus/27EB-4722-03FA
3700    services/6F81-5844-456A/skus/27EC-7D5B-D560
3701    services/6F81-5844-456A/skus/27F2-8847-8993
3702    services/6F81-5844-456A/skus/27F4-5725-4650
3703    services/6F81-5844-456A/skus/27F5-ABD6-7A03
Name: name, Length: 3704, dtype: object